# Chasing the Aha Moment — four-response GRPO pilot
This is the recorded training workflow. It samples **four responses per training question** for **up to 20 optimizer steps** (normally 20 question groups and 80 completions), using Qwen2.5-1.5B-Instruct, the historical `v2-answer-field` reward rules, the fixed system prompt and a 512-token completion cap. It starts a fresh adapter; it does not continue a previous run.

A **20-minute soft training budget** requests a stop at the end of the current optimizer step. It can overrun by one step plus saving time. Installation, downloads and model setup are outside that budget. If the timer fires before step 20, the save and summary cells still run.

1. Open the notebook in a fresh Colab T4 session.
2. Run installation, then select **Runtime → Restart session**.
3. Skip installation and run every remaining cell in order. No manual edits are needed.
4. The run folder printed at the end contains `training_responses.jsonl`, `training.json` and `diagnostics_summary.json`.

The recorded run is in `results/pilot/`. It completed 20 steps in 794 seconds on a Tesla T4 with 3.21 GiB peak allocated memory.

This pilot measures reward variation and optimization activity, not held-out accuracy; it generates no evaluation answers. A changed generation count alone cannot establish that four responses are better than two; that would require a controlled comparison.


In [ ]:
%pip install "transformers==4.56.2" "trl==0.22.2" "peft==0.17.1" "accelerate==1.10.1" "datasets==4.1.1" "bitsandbytes==0.48.1"
# Restart session now, then continue below. Retain Colab's CUDA-enabled PyTorch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, time, subprocess, pathlib, hashlib
from datetime import datetime, timezone
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU'
print(subprocess.check_output(['nvidia-smi']).decode())
RUN = pathlib.Path('/content/drive/MyDrive/aha_pilot_v4') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN.mkdir(parents=True, exist_ok=False)
CFG = dict(model='Qwen/Qwen2.5-1.5B-Instruct', seed=3407, steps=20, train_size=128,
           dev_size=8, max_prompt_length=384, max_completion_length=512, rank=16, scorer_version="v2-answer-field",
           num_generations=4, training_budget_seconds=1200)
(RUN/'config.json').write_text(json.dumps(CFG, indent=2))
(RUN/'environment.txt').write_text(subprocess.check_output(['pip','freeze']).decode())
print('All results and checkpoints:', RUN)

In [ ]:
from pathlib import Path
Path('/content/rewards.py').write_text('"""V2: numeric correctness independent of strict formatting. No GPU required."""\nimport re\nfrom decimal import Decimal, InvalidOperation\nSYSTEM_PROMPT = \'Solve the problem. Respond with <reasoning>your solution</reasoning> followed by <answer>the numeric answer</answer>.\'\nSCORER_VERSION = \'v2-answer-field\'\nPATTERN = re.compile(r\'\\s*<reasoning>(.*?)</reasoning>\\s*<answer>(.*?)</answer>\\s*\', re.S)\nTAGS = (\'<reasoning>\', \'</reasoning>\', \'<answer>\', \'</answer>\')\ndef parse(text):\n    """Strict full-response format, used only for format scoring."""\n    if not isinstance(text, str) or any(text.count(t) != 1 for t in TAGS):\n        return None\n    m = PATTERN.fullmatch(text)\n    if not m or not all(x.strip() for x in m.groups()):\n        return None\n    return m.group(1).strip(), m.group(2).strip()\ndef number(text):\n    """Decimal number; allow a single leading currency symbol and valid commas."""\n    text = str(text).strip()\n    if text[:1] in (\'$\', \'£\', \'€\'):\n        text = text[1:].strip()\n    if not re.fullmatch(r\'[+-]?(?:\\d+|\\d{1,3}(?:,\\d{3})+)(?:\\.\\d+)?\', text):\n        return None\n    try:\n        n = Decimal(text.replace(\',\', \'\'))\n        return n if n.is_finite() else None\n    except InvalidOperation:\n        return None\ndef extract_answer(text):\n    """Require one closed answer field; never infer from intermediate numbers."""\n    if not isinstance(text, str):\n        return None\n    if text.count(\'<answer>\') != 1 or text.count(\'</answer>\') != 1:\n        return None\n    m = re.search(r\'<answer>(.*?)</answer>\', text, re.S)\n    if not m:\n        return None\n    # Permit whitespace/Markdown emphasis after the final field, not extra claims.\n    if m.string[m.end():].strip(\' \\t\\r\\n*_\'):\n        return None\n    return m.group(1).strip()\ndef score(text, gold):\n    answer = extract_answer(text)\n    predicted = number(answer) if answer is not None else None\n    target = number(gold)\n    valid_format = parse(text) is not None\n    correct = predicted is not None and target is not None and predicted == target\n    return dict(format_valid=valid_format, answer_parseable=predicted is not None,\n                correct=correct, structured_correct=correct and valid_format)\ndef contents(completions):\n    return [x if isinstance(x, str) else x[0][\'content\'] for x in completions]\ndef correctness_reward(completions, answer, **kwargs):\n    texts = contents(completions)\n    if len(texts) != len(answer):\n        raise ValueError(\'Completion/answer lengths differ\')\n    return [2.0 * score(t, a)[\'correct\'] for t, a in zip(texts, answer)]\ndef format_reward(completions, **kwargs):\n    return [0.5 if parse(t) else 0.0 for t in contents(completions)]\ndef gold_answer(solution):\n    if \'####\' not in solution:\n        raise ValueError(\'Missing GSM8K answer delimiter\')\n    gold = solution.rsplit(\'####\', 1)[1].strip()\n    if number(gold) is None:\n        raise ValueError(f\'Unsupported gold answer: {gold!r}\')\n    return gold\n')
Path('/content/test_rewards.py').write_text("import unittest\nfrom rewards import *\ndef wrap(a='42', r='First line\\nSecond line'):\n    return f'<reasoning>{r}</reasoning>\\n<answer>\\n{a}\\n</answer>'\nclass RewardsTest(unittest.TestCase):\n    def test_multiline(self): self.assertTrue(score(wrap(), '42')['structured_correct'])\n    def test_plain(self): self.assertFalse(score('42', '42')['correct'])\n    def test_missing_close(self): self.assertIsNone(extract_answer('<answer>42'))\n    def test_missing_open(self): self.assertIsNone(extract_answer('42</answer>'))\n    def test_duplicate(self): self.assertFalse(score(wrap()+'<answer>42</answer>', '42')['correct'])\n    def test_trailing(self): self.assertFalse(score(wrap()+' actually 43', '42')['correct'])\n    def test_empty(self): self.assertIsNone(parse(wrap(r=' ')))\n    def test_wrong(self): self.assertFalse(score(wrap('41'), '42')['correct'])\n    def test_equivalent(self): self.assertTrue(score(wrap('1,200.0'), '1200')['correct'])\n    def test_negative(self): self.assertTrue(score(wrap('-42'), '-42')['correct'])\n    def test_malformed_number(self): self.assertIsNone(number('12,00'))\n    def test_nan(self): self.assertIsNone(number('NaN'))\n    def test_units(self): self.assertIsNone(number('42 dollars'))\n    def test_gold(self): self.assertEqual(gold_answer('work\\n#### 42'), '42')\n    def test_batch(self): self.assertEqual(correctness_reward([[{'content':wrap()}]], ['42']), [2.0])\n    def test_length_mismatch(self):\n        with self.assertRaises(ValueError): correctness_reward([wrap()], [])\n    def test_independent(self):\n        s=score('Some work\\n<answer>87</answer>', '87')\n        self.assertTrue(s['correct']); self.assertFalse(s['format_valid'])\n    def test_currency(self): self.assertTrue(score('<answer>$15</answer>', '15')['correct'])\n    def test_markdown(self): self.assertTrue(score('**<answer>58</answer>**', '58')['correct'])\n    def test_multiple_numbers(self): self.assertFalse(score('<answer>42 or 43</answer>', '42')['correct'])\n    def test_nested(self): self.assertFalse(score('<answer><answer>42</answer></answer>', '42')['correct'])\n    def test_intermediate(self): self.assertFalse(score('42 is intermediate; final result unknown', '42')['correct'])\n    def test_mismatched_order(self): self.assertIsNone(extract_answer('</answer>42<answer>'))\n    def test_incomplete_no_reward(self): self.assertEqual(correctness_reward(['<answer>4'],['4']),[0.0])\nif __name__ == '__main__': unittest.main()\n")
Path('/content/diagnostics.py').write_text('"""Record reward inputs without changing the v2 reward values."""\nimport json, os\nfrom collections import defaultdict\nfrom pathlib import Path\nfrom rewards import contents, correctness_reward, format_reward, extract_answer, number, score\n\ndef make_logged_reward(path, max_tokens=512):\n    path = Path(path)\n    if path.exists():\n        raise FileExistsError(\'Diagnostic log already exists. Start a new run instead of mixing runs.\')\n    path.touch()\n    call_id = 0\n    def logged_correctness_reward(prompts, completions, answer, completion_ids=None,\n                                  source_id=None, trainer_state=None, **kwargs):\n        nonlocal call_id\n        texts = contents(completions)\n        n = len(texts)\n        if len(prompts) != n or len(answer) != n:\n            raise ValueError(\'Prompt/completion/answer lengths differ\')\n        if source_id is not None and len(source_id) != n:\n            raise ValueError(\'source_id length mismatch\')\n        if completion_ids is not None and len(completion_ids) != n:\n            raise ValueError(\'completion_ids length mismatch\')\n        rewards = correctness_reward(completions, answer)\n        formats = format_reward(completions)\n        call_id += 1\n        rows = []\n        for i, (prompt, text, gold) in enumerate(zip(prompts, texts, answer)):\n            extracted = extract_answer(text)\n            numeric = number(extracted) if extracted is not None else None\n            result = score(text, gold)\n            if extracted is None:\n                status = \'missing_or_ambiguous_answer_field\'\n            elif numeric is None:\n                status = \'unsupported_numeric_answer\'\n            elif not result[\'correct\']:\n                status = \'parsed_answer_mismatch\'\n            else:\n                status = \'correct\'\n            tokens = len(completion_ids[i]) if completion_ids is not None else None\n            rows.append(dict(call_id=call_id, completion_index=i,\n                optimizer_steps_completed=getattr(trainer_state, \'global_step\', None),\n                source_id=int(source_id[i]) if source_id is not None else None,\n                prompt=prompt, gold=str(gold), response=text, extracted_answer=extracted,\n                normalized_answer=str(numeric) if numeric is not None else None,\n                status=status, generated_tokens=tokens,\n                hit_token_cap=tokens >= max_tokens if tokens is not None else None,\n                correctness_reward=rewards[i], format_reward=formats[i],\n                total_reward=rewards[i]+formats[i], **result))\n        with path.open(\'a\') as f:\n            for row in rows: f.write(json.dumps(row)+\'\\n\')\n            f.flush(); os.fsync(f.fileno())\n        return rewards\n    return logged_correctness_reward\n\ndef summarize(path):\n    rows=[json.loads(l) for l in Path(path).read_text().splitlines() if l.strip()]\n    groups=defaultdict(list)\n    counts=defaultdict(int)\n    for r in rows:\n        groups[(r[\'call_id\'], json.dumps(r[\'prompt\'], sort_keys=True))].append(r[\'total_reward\'])\n        counts[r[\'status\']]+=1\n    return dict(logged_completions=len(rows), reward_calls=len({r[\'call_id\'] for r in rows}),\n        prompt_groups=len(groups), groups_with_reward_variation=sum(len(set(v))>1 for v in groups.values()),\n        status_counts=dict(counts), format_valid=sum(r[\'format_valid\'] for r in rows),\n        hit_token_cap=sum(r[\'hit_token_cap\'] is True for r in rows),\n        note=\'Counts describe sampled training responses, not held-out accuracy. Missing answer fields need human inspection.\')\n')
Path('/content/test_diagnostics.py').write_text("import json, tempfile, unittest\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom diagnostics import make_logged_reward, summarize\nfrom rewards import correctness_reward\nclass LoggingTests(unittest.TestCase):\n def setUp(self):\n  self.tmp=tempfile.TemporaryDirectory(); self.addCleanup(self.tmp.cleanup)\n  self.p=Path(self.tmp.name)/'log.jsonl'\n def test_preserves_rewards_and_logs(self):\n  f=make_logged_reward(self.p)\n  texts=['<answer>42</answer>','<answer>41</answer>']\n  result=f(prompts=['q','q'], completions=texts, answer=['42','42'],source_id=[9,9],completion_ids=[[1],[2]],trainer_state=SimpleNamespace(global_step=0))\n  self.assertEqual(result,correctness_reward(texts,['42','42']))\n  rows=[json.loads(l) for l in self.p.read_text().splitlines()]\n  self.assertEqual(rows[0]['source_id'],9);self.assertEqual(rows[0]['optimizer_steps_completed'],0)\n  summary=summarize(self.p);self.assertEqual(summary['groups_with_reward_variation'],1)\n def test_no_overwrite(self):\n  self.p.write_text('original')\n  with self.assertRaises(FileExistsError):make_logged_reward(self.p)\n  self.assertEqual(self.p.read_text(),'original')\n def test_failed_parse_is_not_math_failure(self):\n  f=make_logged_reward(self.p);f(['q'],['Final answer: 42'],['42'])\n  self.assertEqual(summarize(self.p)['status_counts'],{'missing_or_ambiguous_answer_field':1})\n def test_mismatch_fails(self):\n  f=make_logged_reward(self.p)\n  with self.assertRaises(ValueError):f([],['x'],['42'])\n def test_multiple_calls(self):\n  f=make_logged_reward(self.p)\n  for _ in range(2):f(['q'],['<answer>1</answer>'],['1'])\n  self.assertEqual(summarize(self.p)['reward_calls'],2)\n def test_format_reward_and_tokens(self):\n  f=make_logged_reward(self.p,max_tokens=2)\n  f(['q'],['<reasoning>x</reasoning><answer>1</answer>'],['1'],completion_ids=[[1,2]])\n  row=json.loads(self.p.read_text());self.assertEqual(row['total_reward'],2.5)\n  self.assertTrue(row['hit_token_cap'])\nif __name__=='__main__':unittest.main()\n")
import sys
sys.path.insert(0, '/content')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', '/content', '-p', 'test_*.py', '-v'], check=True)
from rewards import *
(RUN/'rewards.py').write_text(Path('/content/rewards.py').read_text())
from diagnostics import make_logged_reward, summarize
(RUN/'diagnostics.py').write_text(Path('/content/diagnostics.py').read_text())


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
set_seed(CFG['seed'])
tokenizer = AutoTokenizer.from_pretrained(CFG['model'])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'
raw = load_dataset('openai/gsm8k', 'main', split='train')
# Reserve development examples from TRAIN. The official TEST split remains untouched.
raw = raw.add_column('source_id', list(range(len(raw))))
split = raw.train_test_split(test_size=128, seed=CFG['seed'])
def prepare(x):
    return dict(prompt=[{'role':'system','content':SYSTEM_PROMPT}, {'role':'user','content':x['question']}],
                answer=gold_answer(x['answer']))
train = split['train'].map(prepare)
dev = split['test'].map(prepare)
def fits(x):
    return len(tokenizer.apply_chat_template(x['prompt'], tokenize=True, add_generation_prompt=True)) <= CFG['max_prompt_length']
train = train.filter(fits).select(range(CFG['train_size']))
dev = dev.filter(fits).select(range(CFG['dev_size']))
assert set(train['source_id']).isdisjoint(dev['source_id'])
(RUN/'data_manifest.json').write_text(json.dumps(dict(dataset='openai/gsm8k', split='train',
    fingerprint=raw._fingerprint, train_ids=list(train['source_id']), dev_ids=list(dev['source_id']),
    system_prompt=SYSTEM_PROMPT, development_only=True), indent=2))
train.to_json(str(RUN/'train.jsonl'))
dev.to_json(str(RUN/'dev.jsonl'))
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(CFG['model'], quantization_config=quant,
    device_map={'':0}, dtype=torch.float16, attn_implementation='sdpa')
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=CFG['rank'], lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj'], lora_dropout=0.0,
    bias='none', task_type='CAUSAL_LM'))
model.config.use_cache = False
model.print_trainable_parameters()
(RUN/'model_revision.json').write_text(json.dumps({'model': CFG['model'],
    'resolved_commit': getattr(model.config, '_commit_hash', None)}, indent=2))

## Record the training evidence
The following cell prepares logging; it does not generate answers. Each reward call records the actual question/prompt, source ID, reference answer, sampled output, extracted number, separate rewards, completion length, and completed optimizer-step count. Files are flushed after every reward call.

A missing or unsupported answer field is reported as an extraction failure, not automatically as incorrect math. Inspect those responses manually. Logging leaves reward values unchanged. The final cell checks actual LoRA parameter differences, independently of displayed loss.


In [ ]:
logged_reward = make_logged_reward(RUN/'training_responses.jsonl', CFG['max_completion_length'])
print('Training-response logging ready. Run the next cell.')


In [ ]:
from trl import GRPOConfig, GRPOTrainer

class CompatibleGRPOConfig(GRPOConfig):
    def __post_init__(self):
        # TRL 0.22.2 copies an initialized config in prepare_peft_model.
        # Clear the derived field so its parent can recompute it consistently.
        if self.generation_batch_size is not None and self.steps_per_generation is not None:
            self.steps_per_generation = None
        super().__post_init__()

from transformers import TrainerCallback

class TrainingTimeBudget(TrainerCallback):
    def __init__(self, seconds, clock=time.perf_counter):
        self.seconds = seconds
        self.clock = clock
        self.started = None
        self.triggered = False
    def on_train_begin(self, args, state, control, **kwargs):
        self.started = self.clock()
        return control
    def on_step_end(self, args, state, control, **kwargs):
        if self.started is not None and self.clock() - self.started >= self.seconds:
            self.triggered = True
            control.should_training_stop = True
            control.should_save = True
        return control

budget = TrainingTimeBudget(CFG['training_budget_seconds'])

args=CompatibleGRPOConfig(output_dir=str(RUN/'checkpoints'), learning_rate=5e-6,
    per_device_train_batch_size=1, gradient_accumulation_steps=CFG['num_generations'],
    num_generations=CFG['num_generations'], max_prompt_length=CFG['max_prompt_length'],
    max_completion_length=CFG['max_completion_length'], max_steps=CFG['steps'],
    save_strategy='steps', save_steps=1, save_total_limit=2, logging_steps=1,
    fp16=True, bf16=False, gradient_checkpointing=True,
    optim='adamw_torch', report_to='none', seed=CFG['seed'], data_seed=CFG['seed'],
    use_vllm=False, beta=0.0, temperature=0.8)
# Effective generation batch = 1 x 4 accumulation: four responses per question.
# This replaces the old five overlapping rewards with correctness (2) + format (0.5).
trainer=GRPOTrainer(model=model, processing_class=tokenizer, args=args,
    reward_funcs=[logged_reward, format_reward], train_dataset=train, callbacks=[budget])
# TRL re-prepares the already wrapped quantized model and freezes parameters.
# Restore just the existing LoRA matrices before optimizer creation.
for name, parameter in trainer.model.named_parameters():
    parameter.requires_grad_("lora_" in name)
trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
assert trainable > 0, "No trainable LoRA parameters found"
assert trainer.args.generation_batch_size % trainer.args.num_generations == 0
assert trainer.args.steps_per_generation is not None
print(f"Trainable LoRA parameters: {trainable:,}")
trainer.model.train()
(RUN/'trainer_config.json').write_text(trainer.args.to_json_string())
# Write the actual installed versions, including the CUDA compatibility fix.
(RUN/'environment.txt').write_text(subprocess.check_output(['pip','freeze']).decode())
# CPU snapshot is roughly 17 MB for this configuration; no second GPU model.
adapter_before = {name: parameter.detach().float().cpu().clone()
                  for name, parameter in trainer.model.named_parameters()
                  if parameter.requires_grad}
assert adapter_before, 'No adapter parameters to compare'
torch.cuda.reset_peak_memory_stats()
start=time.perf_counter()
result=trainer.train()
trainer.save_model(str(RUN/'adapter'))
tokenizer.save_pretrained(str(RUN/'adapter'))
trainer.save_state()
(RUN/'training.json').write_text(json.dumps(dict(metrics=result.metrics,
    seconds=time.perf_counter()-start,
    peak_allocated_gib=torch.cuda.max_memory_allocated()/2**30,
    peak_reserved_gib=torch.cuda.max_memory_reserved()/2**30,
    history=trainer.state.log_history, scorer_version=SCORER_VERSION,
    requested_steps=CFG['steps'], completed_steps=trainer.state.global_step,
    training_budget_seconds=CFG['training_budget_seconds'],
    budget_triggered=budget.triggered,
    stop_reason='step_limit' if trainer.state.global_step >= CFG['steps'] else
        ('time_budget' if budget.triggered else 'other')), indent=2))
print('Adapter and training state saved:', RUN)

In [ ]:
import math
per_tensor = []
for name, parameter in trainer.model.named_parameters():
    if name not in adapter_before:
        continue
    delta = parameter.detach().float().cpu() - adapter_before[name]
    assert torch.isfinite(delta).all().item(), f'Nonfinite adapter change: {name}'
    per_tensor.append(dict(name=name, changed_elements=int(torch.count_nonzero(delta).item()),
        max_abs_change=float(delta.abs().max().item()), squared_l2=float(delta.double().square().sum().item())))
assert len(per_tensor) == len(adapter_before), 'Adapter names changed during setup'
summary = summarize(RUN/'training_responses.jsonl')
summary['scorer_version'] = SCORER_VERSION
summary['num_generations'] = CFG['num_generations']
summary['requested_optimizer_steps'] = CFG['steps']
summary['budget_triggered'] = budget.triggered
summary['stop_reason'] = 'step_limit' if trainer.state.global_step >= CFG['steps'] else ('time_budget' if budget.triggered else 'other')
summary['fraction_groups_with_reward_variation'] = (summary['groups_with_reward_variation'] / summary['prompt_groups'] if summary['prompt_groups'] else None)
logged_rows = [json.loads(line) for line in (RUN/'training_responses.jsonl').read_text().splitlines() if line.strip()]
summary['unique_training_questions'] = len({row['source_id'] for row in logged_rows})
summary['mean_reward'] = sum(row['total_reward'] for row in logged_rows)/len(logged_rows) if logged_rows else None
summary['completed_optimizer_steps'] = trainer.state.global_step
summary['steps_with_nonzero_gradient'] = sum(
    r.get('grad_norm', 0) > 0 for r in trainer.state.log_history)
summary['adapter_change'] = dict(
    tensors_checked=len(per_tensor),
    tensors_changed=sum(x['changed_elements'] > 0 for x in per_tensor),
    elements_changed=sum(x['changed_elements'] for x in per_tensor),
    l2_change=math.sqrt(sum(x['squared_l2'] for x in per_tensor)),
    max_abs_change=max(x['max_abs_change'] for x in per_tensor))
summary['adapter_change']['weights_changed'] = summary['adapter_change']['elements_changed'] > 0
summary['interpretation'] = 'Weight changes confirm optimization activity, not improved reasoning or held-out accuracy.'
(RUN/'diagnostics_summary.json').write_text(json.dumps(summary, indent=2))
(RUN/'adapter_deltas.json').write_text(json.dumps(per_tensor, indent=2))
print(json.dumps(summary, indent=2))
print('Finished. Download these THREE files from:', RUN)
for name in ['training_responses.jsonl', 'training.json', 'diagnostics_summary.json']:
    assert (RUN/name).exists(), f'Missing file: {name}'
    print(name)


## After completion
Outputs are under **My Drive → aha_pilot_v4 → newest timestamp folder**. Earlier runs are untouched.

Keep each run as recorded, including any time-budget stop, and do not extend it in place after seeing its rewards. Judge a run by the number of distinct questions, the fraction of groups with differing rewards, parsing failures, truncation, gradients, weight changes and elapsed time — not by training accuracy, which is no substitute for independent evaluation.

More steps change the learning-rate schedule, and four samples change random-number consumption, so comparison with earlier two-response runs is exploratory. The official GSM8K test split remains unused.
